# 2.6. Maquinas de Soporte Vectorial

## Setup

In [1]:
import math
import os
import warnings
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import random
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import plotly.figure_factory as ff
import session_info
from scipy import stats
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from sklearn import set_config
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, confusion_matrix,make_scorer,recall_score)
from sklearn.model_selection import (train_test_split, RepeatedKFold, RepeatedStratifiedKFold, 
                                     cross_validate, cross_val_score, KFold)
from sklearn import datasets

# Configuraciones
sns.set_style('dark')
pd.set_option("display.max_columns", 100)
pio.templates.default = "plotly_dark"
set_config(transform_output="pandas")
set_config(display='diagram')
warnings.filterwarnings("ignore")

# Magic Commands
%matplotlib inline

# Directorio de datos
dir_data = '../03_Data/'

random.seed(333)

In [2]:
def get_cv_scores_report_classification(estimator, X, y, n_splits):
    cv_scores = cross_validate(
                    estimator = estimator,
                    X         = X,
                    y         = y,
                    scoring   = {
                                'accuracy': make_scorer(accuracy_score), 
                                'recall': make_scorer(recall_score, average='weighted'), 
                                'roc_auc_ovr': 'roc_auc_ovr'
                                },
                    cv        = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=5, random_state=333),
                )
    
    # Convertir el diccionario a dataframe para facilitar la visualización
    cv_scores = pd.DataFrame(cv_scores)
    print(f"Accuracy en CV: mean {cv_scores.test_accuracy.mean():.2f}, std {cv_scores.test_accuracy.std():.2f}")
    print(f"Recall en CV: mean {cv_scores.test_recall.mean():.2f}, std {cv_scores.test_recall.std():.2f}")
def print_data(data):
    for row in data:
        print(''.join('{:3}'.format(value) for value in row))
def flatten_data(data):
    return[np.reshape(data, (28*28,))]
def get_data(number):
    img = Image.open(f"../03_Data/digits_val/{number}.jpeg").convert('L')
    img_arr = np.array(img)
    WIDTH, HEIGHT = img.size
    data = list(img.getdata())
    data = [data[offset:offset+WIDTH] for offset in range(0, WIDTH*HEIGHT, WIDTH)]
    return data

In [3]:
session_info.show()

## 2.6.0. Maquinas de Soporte Vectorial

### Contexto

Dado un conjunto de entrenamiento de puntos etiquetados $(x_1, y_1), (x_2, y_2), \ldots, (x_n, y_n)$ donde $x_i \in \mathbb{R}^p$ y $y_i \in \{-1, 1\}$ (para la clasificación binaria), SVM busca encontrar el hiperplano que mejor separa las dos clases.

Un hiperplano en un espacio p-dimensional está definido por:

$$w^T x + b = 0$$

donde $w \in \mathbb{R}^p$ es el vector normal al hiperplano y $b \in \mathbb{R}$ es un término de sesgo.

La distancia $d$ de un punto $x_0$ al hiperplano se calcula usando la fórmula:
$$ d = \frac{|w \cdot x_0 + b|}{\|w\|} $$

Para una SVC, los vectores de soporte son aquellos puntos que están más cerca del hiperplano de decisión. Si etiquetamos los datos de una clase como +1 y de la otra clase como -1, entonces para los vectores de soporte:
$$ w \cdot x_{+} + b = 1$$ 
para la clase +1
$$ w \cdot x_{-} + b = -1$$ 
para la clase -1

Restando la segunda ecuación de la primera, obtenemos:
$$ w \cdot (x_{+} - x_{-}) = 2 $$

La cantidad $x_{+} - x_{-}$ es la diferencia vectorial entre un vector de soporte de la clase +1 y un vector de soporte de la clase -1. La proyección de esta diferencia en la dirección del vector normal $w$ es el margen. Por lo tanto, la longitud de este margen es:

$$ \frac{w \cdot (x_{+} - x_{-})}{\|w\|} = \frac{2}{\|w\|} $$


**La restricción del margen:**

Queremos que todos los puntos en el conjunto de datos satisfagan la restricción de que, si $y_i = 1$, entonces $w^T x_i + b \geq 1$, y si $y_i = -1$, entonces $w^T x_i + b \leq -1$. Esta restricción asegura que el margen sea al menos 1.

La tarea principal de SVM es maximizar el margen, lo que es equivalente a minimizar $\|w\|$. Así, podemos formular el siguiente problema de optimización convexa:

$$
\begin{align*}
\text{Minimizar:} & \quad \frac{1}{2} \|w\|^2 \\
\text{Sujeto a:} & \quad y_i(w^T x_i + b) \geq 1, \quad i = 1, \ldots, n
\end{align*}
$$

In [143]:
X, y = datasets.make_blobs(n_samples=50, n_features=2, centers=2, cluster_std=1.05, random_state=40)
clf = SVC(kernel='linear')
clf.fit(X, y)
w = clf.coef_
b = clf.intercept_

# Calcular los puntos del hiperplano y del margen
x_plot = np.linspace(X[:, 0].min(), X[:, 0].max(), 100)
y_plot_decision = -(w[0][0]*x_plot + b)/w[0][1]
y_plot_margin_pos = -(w[0][0]*x_plot + b - 1)/w[0][1]
y_plot_margin_neg = -(w[0][0]*x_plot + b + 1)/w[0][1]

fig = go.Figure()
fig.add_trace(go.Scatter(x=X[y==1][:, 0], y=X[y==1][:, 1], mode='markers', name='Clase 1'))
fig.add_trace(go.Scatter(x=X[y==0][:, 0], y=X[y==0][:, 1], mode='markers', name='Clase 0'))
fig.add_trace(go.Scatter(x=x_plot, y=y_plot_decision, mode='lines', name='Hiperplano de decisión'))
fig.add_trace(go.Scatter(x=x_plot, y=y_plot_margin_pos, mode='lines', line=dict(dash='dot'), name='Margen Positivo'))
fig.add_trace(go.Scatter(x=x_plot, y=y_plot_margin_neg, mode='lines', line=dict(dash='dot'), name='Margen Negativo'))

fig.show()

En muchos casos prácticos, los datos no son linealmente separables. Para manejar estos casos, introducimos un **soft margin** permitiendo que algunas instancias estén dentro del margen o incluso en el lado incorrecto del hiperplano. Introducimos variables de holgura $\xi_i \geq 0$:

$$
\begin{align*}
\text{Minimizar:} & \quad \frac{1}{2} \|w\|^2 + C \sum_{i=1}^n \xi_i \\
\text{Sujeto a:} & \quad y_i(w^T x_i + b) \geq 1 - \xi_i, \quad i = 1, \ldots, n \\
& \quad \xi_i \geq 0, \quad i = 1, \ldots, n
\end{align*}
$$

Aquí, $C$ es un parámetro de regularización.

In [152]:
X, y = datasets.make_blobs(n_samples=100, n_features=2, centers=2, cluster_std=2, random_state=33)
y = 2*y - 1

# Entrenar un Soft Margin SVM
clf = SVC(kernel='linear', C=0.5)
clf.fit(X, y)
w = clf.coef_
b = clf.intercept_

# Calcular los puntos del hiperplano y del margen
x_plot = np.linspace(X[:, 0].min(), X[:, 0].max(), 100)
y_plot_decision = -(w[0][0]*x_plot + b)/w[0][1]
y_plot_margin_pos = -(w[0][0]*x_plot + b - 1)/w[0][1]
y_plot_margin_neg = -(w[0][0]*x_plot + b + 1)/w[0][1]

fig = go.Figure()
fig.add_trace(go.Scatter(x=X[y==1][:, 0], y=X[y==1][:, 1], mode='markers', name='Clase 1'))
fig.add_trace(go.Scatter(x=X[y==-1][:, 0], y=X[y==-1][:, 1], mode='markers', name='Clase -1'))
fig.add_trace(go.Scatter(x=x_plot, y=y_plot_decision, mode='lines', name='Hiperplano de decisión'))
fig.add_trace(go.Scatter(x=x_plot, y=y_plot_margin_pos, mode='lines', line=dict(dash='dot'), name='Margen Positivo'))
fig.add_trace(go.Scatter(x=x_plot, y=y_plot_margin_neg, mode='lines', line=dict(dash='dot'), name='Margen Negativo'))
fig.show()


Para resolver el problema de optimización anterior, introducimos multiplicadores de Lagrange $ \alpha_i $ (uno para cada restricción) y $ \mu_i $ (para las restricciones de las variables de holgura). La función lagrangiana $ L $ es:

$$
L(w, b, \xi, \alpha, \mu) = \frac{1}{2} \|w\|^2 + C \sum_{i=1}^n \xi_i - \sum_{i=1}^n \alpha_i [y_i(w^T x_i + b) - 1 + \xi_i] - \sum_{i=1}^n \mu_i \xi_i
$$

Donde $ \alpha_i \geq 0 $ y $ \mu_i \geq 0 $ son los multiplicadores de Lagrange.

Para encontrar el mínimo de la función lagrangiana con respecto a $ w $, $ b $ y $ \xi $, derivamos $ L $ y establecemos las derivadas a cero:

$$
\begin{align*}
\frac{\partial L}{\partial w} & = w - \sum_{i=1}^n \alpha_i y_i x_i = 0 \implies w = \sum_{i=1}^n \alpha_i y_i x_i \\
\frac{\partial L}{\partial b} & = -\sum_{i=1}^n \alpha_i y_i = 0 \\
\frac{\partial L}{\partial \xi_i} & = C - \alpha_i - \mu_i = 0
\end{align*}
$$

**<span style="color:red">R1</span>** Sustituyendo las condiciones anteriores de vuelta en la función lagrangiana, obtenemos el problema dual:

$$
\begin{align*}
\text{Maximizar:} & \quad \sum_{i=1}^n \alpha_i - \frac{1}{2} \sum_{i,j=1}^n \alpha_i \alpha_j y_i y_j x_i^T x_j \\
\text{Sujeto a:} & \quad \sum_{i=1}^n \alpha_i y_i = 0 \\
& \quad 0 \leq \alpha_i \leq C, \quad i = 1, \ldots, n
\end{align*}
$$

Para datos que no son linealmente separables, podemos mapear los datos a un espacio de dimensiones más altas donde sí lo sean. Este mapeo se realiza usando funciones **kernel** como lo vimos en **regresion ridge kernel**.

Si $K(x, x')$ es una función de kernel, el problema de optimización se basa en los datos mapeados en lugar de los puntos de datos directamente.

En este nuevo problema de optimización se tiene una forma dual que es útil para kernels. La solución dual se basa en los multiplicadores de Lagrange.

### Ejemplo

**Digitos**

El conjunto de datos.

In [5]:
28*28

784

In [17]:
flatten_data((get_data(3)))

[array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  15,   1,   0,
          0,   4,   0,   6,   0,  11,   3,   0,   0,   0,   3,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  18,   7,   0,
          0,   5,  16,   0,  10,   0,  27,   0,   0,   0,  20,   4,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,  15,   1,   0,   4,  12,   2,  10,   0,  13,   9,   0,   2,
          0,   3,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,  19,   0,   0,  12,   0,   1,   0,   0,   0,   6,   0,   0,
          2,   7,   0,  10,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,  10, 250, 255, 240, 250, 255, 255, 255, 247,
        255,  14,   0,   0,   5,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   6,   2, 255, 255, 255, 255, 238, 255,
        240, 255, 247,  12,   0,   0,  10,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   1

In [7]:
print_data((get_data(9)))

  0  0  0  0  0  0  0  0  0  0  3  0  0 10  0  5  0 11  0  8  0  0 16  0  0  0  0  0
  0  0  0  0  0  0  0  0  5  0  6  5  0  0  9  0 19  0 13  0  6  4  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0 18  0  0 21  0  4 11  0  5  0  0  0  0  2  9  0  0  0  0
  0  0  0  0  0  0  0  0  6  4  0  5  1  6  0  4 12  0  2 10 22  0  8  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  8  0  0255255255248 15  0  0  0  5  0 15  0  0  0  0
  0  0  0  0  0  0  0  0  0 10  0255255253255237247255255  1 20  0  0 12  0  0  0  0
  0  0  0  0  0  0  0  0  4245252251243255 12  9255247255255  0 18  2  0  0  0  0  0
  0  0  0  0  0  0  0  0254255255248  8  5  0  0  0  6247255  6  0  0 13  0  0  0  0
  0  0  0  0  0  0  0  0250255  4  0  0  6  0  3  1 12255239 10  4  2  0  0  0  0  0
  0  0  0  0  0  0  0  0255255  0  7  1  0  1  0  0  0  0255  1  0  0 12  0  0  0  0
  0  0  0  0  0  0  0  0252232 14  0  0 14  0 22  0 16 15250  0  7  0  3  0  0  0  0
  0  0  0  0  0  0  0  0255255  0  0  4 16  0  0 15  0250237 20  

#### Manipulación de Datos y AED

In [8]:
df = pd.read_csv('../03_Data/digits_val/digits_train_sample.csv')
df.head()

,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,1x10,1x11,1x12,1x13,1x14,1x15,1x16,1x17,1x18,1x19,1x20,1x21,1x22,1x23,1x24,1x25,1x26,1x27,1x28,2x1,2x2,2x3,2x4,2x5,2x6,2x7,2x8,2x9,2x10,2x11,2x12,2x13,2x14,2x15,2x16,2x17,2x18,2x19,2x20,2x21,...,27x7,27x8,27x9,27x10,27x11,27x12,27x13,27x14,27x15,27x16,27x17,27x18,27x19,27x20,27x21,27x22,27x23,27x24,27x25,27x26,27x27,27x28,28x1,28x2,28x3,28x4,28x5,28x6,28x7,28x8,28x9,28x10,28x11,28x12,28x13,28x14,28x15,28x16,28x17,28x18,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [89]:
df.shape

(6000, 785)

In [90]:
label_counts = df["label"].value_counts()
fig = px.pie(values=label_counts, names=label_counts.index, title="Distribución de Dígitos")
fig.show()

In [10]:
# Aplicamos Análisis de Componentes Principales
pca = PCA(n_components=0.9)
pca.fit(df[[x for x in df.columns if x != "label"]])

PCA(n_components=0.9)

In [92]:
print('Pasamos de', df.shape[1],'características a', pca.n_components_, 'componentes con el', np.round(np.sum(pca.explained_variance_ratio_)* 100,2),"% de varianza explicada.")

Pasamos de 785 características a 86 componentes con el 90.04 % de varianza explicada.


In [93]:
Xp = pca.transform(df[[x for x in df.columns if x != "label"]])
Xp["label"] = df["label"].astype(str)
Xp.head()

,pca0,pca1,pca2,pca3,pca4,pca5,pca6,pca7,pca8,pca9,pca10,pca11,pca12,pca13,pca14,pca15,pca16,pca17,pca18,pca19,pca20,pca21,pca22,pca23,pca24,pca25,pca26,pca27,pca28,pca29,pca30,pca31,pca32,pca33,pca34,pca35,pca36,pca37,pca38,pca39,pca40,pca41,pca42,pca43,pca44,pca45,pca46,pca47,pca48,pca49,pca50,pca51,pca52,pca53,pca54,pca55,pca56,pca57,pca58,pca59,pca60,pca61,pca62,pca63,pca64,pca65,pca66,pca67,pca68,pca69,pca70,pca71,pca72,pca73,pca74,pca75,pca76,pca77,pca78,pca79,pca80,pca81,pca82,pca83,pca84,pca85,label
0,-520.692335,-364.422286,-216.328248,-162.579688,-464.573050,120.849085,-146.763627,38.388834,-308.221880,-21.691646,56.439949,156.546914,50.288325,-206.246626,-90.370558,73.562830,-71.519077,409.440061,84.976306,-114.642590,-200.097876,277.253523,-16.735968,69.349668,-10.644849,355.420866,-210.306764,121.730439,167.500936,82.530040,32.609327,-54.124549,-47.779863,19.311100,-36.777512,-1.940550,53.614423,39.484260,-97.898974,-62.894944,195.918708,118.097409,-52.558044,9.359503,-130.029115,73.620469,-5.864432,-131.776409,40.165347,-245.619149,80.628287,-93.110451,-121.676022,-62.349797,97.781244,-50.715963,-36.680422,-8.921881,209.442533,-91.305064,-4.714470,-60.620116,86.774715,24.114844,-123.557110,160.651151,-57.937424,79.736782,-164.454787,123.523508,37.324011,86.572605,15.192405,-16.258569,33.676741,44.393170,147.292454,38.012708,-105.909579,79.857371,114.571611,171.027425,-48.842454,3.969150,100.301714,-61.483159,3
1,-54.585336,141.895235,-753.895660,421.860756,-536.536276,-198.004906,-232.394046,-44.681933,3.436320,381.576694,161.720499,-196.187902,43.739839,322.006632,206.245402,-364.136810,26.957492,-22.955334,-168.893080,307.308203,-96.262131,-178.287068,-9.534546,201.478832,-130.926617,446.035275,-107.675796,-141.840632,91.368346,355.035637,-38.571890,-97.004145,-1.476252,295.439643,29.594501,-107.809126,65.666014,-78.762409,182.354494,133.577400,144.208847,-85.791106,-122.858597,-84.520344,21.258295,148.748460,-80.581165,-21.746632,-145.575186,-49.682652,9.996832,-179.376230,-138.116958,57.678710,10.636200,26.341633,-104.709412,28.882446,33.612140,70.873134,-77.083294,75.585736,9.596740,168.152441,-78.545659,-92.341008,-28.823534,-5.945370,171.879727,22.916847,-17.083767,-34.024232,-13.236979,-58.222983,58.620448,-148.881223,31.820116,-80.220621,47.027505,106.242375,1.094981,19.574749,-161.598783,33.084577,26.721045,-67.312477,5
2,-477.532315,85.148804,-769.669313,121.069730,-316.283616,-305.196222,-147.438070,293.763466,-106.114388,-152.709653,230.827360,-74.706332,-251.597697,232.417987,119.027003,-315.741513,-63.099293,127.729474,-134.811476,45.529648,117.637574,40.451580,-56.451437,54.570030,-229.214186,-53.102750,-12.014349,-69.860298,166.840691,-331.630036,18.086625,82.143077,270.774028,-201.178981,157.815943,-0.293453,-41.033537,-220.971160,156.212872,-177.592829,55.793284,-107.382385,56.256870,-136.410130,21.532252,16.030610,77.206409,15.185252,176.234220,113.637454,-108.282298,-75.920373,80.541454,-76.624985,69.831397,-174.347523,146.146725,55.836344,-16.394361,223.925396,65.706969,49.538880,46.710234,-221.911702,34.326986,14.254554,190.719840,-2.611327,90.289982,-7.022128,-5.155446,-116.971440,-45.050381,-39.973196,-33.761780,-10.412244,-97.679995,0.853704,-27.530519,-141.680053,-118.389069,40.101680,100.899733,65.058813,-129.891155,-56.024686,3
3,-193.004343,-215.973609,-656.619268,-202.091937,447.300048,-351.943955,206.908797,-356.798588,401.003098,-412.036825,-1.784006,-387.030696,-163.569613,347.622841,17.919018,231.841942,38.323016,-25.080761,-251.618188,-97.887449,-85.190394,-266.947858,-100.743474,23.148438,78.051454,137.865448,305.363899,133.294162,8.228677,-211.260208,-81.356848,-72.655839,59.383093,270.158935,-149.383577,-2.309648,-84.627629,131.983168,-151.059087,-211.476032,-186.000193,-145.251929,-116.568046,-30.684785,-208.976826,-155.186517,-132.552032,-163.165059,166.082192,9.378985,91.854740,109.224251,-8.269188,114.554491,-115.263977,160.498330,-75.804344,49.307344,-102.336804,-184.40

In [94]:
np.round(np.sum(pca.explained_variance_ratio_[:2])* 100)

17.0

In [95]:
fig = px.scatter(Xp, x="pca0", y="pca1", color="label")
fig.show()

In [12]:
# Indica un dígito aquí:
aux = df[df["label"]==4].sample(5).reset_index(drop=True)
for i in range(5):
    vec = aux.loc[i, [x for x in aux.columns if x!="label"]].values
    print_data(vec.reshape((28, 28)))
    print("\n")

  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0 73253207 13  0  0  0  0  0  0  0  5117 76  0  0  0  0
  0  0  0  0  0  0  0  0  0  0130252252 35  0  0  0  0  0  0  7158252206  7  0  0  0
  0  0  0  0  0  0  0  0  0  0 90252252 69  0  0  0  0  0  0 73252252184  5  0  0  0
  0  0  0  0  0  0  0  0  0  0193252252 23  0  0  0  0  0  0153252234 59  0  0  0  0
  0  0  0  0  0  0  0  0  0 15207252229 18  0  0  0  0  0 73235252127  0  0  0  0  0
  0  0  0  0  0  0  0  0  0152252252 64  0  0  0  0  0 72235252252 36  0  0  0  0  0
  0  0  0  0  0  0  0  0 17205252252 36  0  0  0  0 3023725225214

In [13]:
X = df[[x for x in df.columns if x != "label"]]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=.7,stratify=df['label'],random_state=333)

In [14]:
sc_x = MinMaxScaler()
Xs = sc_x.fit_transform(X_train)

#### Modelación

Los hiperparámetros principales en SVC son:

1. **C**: Es el parámetro de regularización. Un valor pequeño de C crea un margen más amplio, lo que puede resultar en más errores de clasificación en los datos de entrenamiento. Un valor grande de C crea un margen más estrecho, lo que puede hacer que el modelo se ajuste demasiado a los datos de entrenamiento. 

2. **Kernel**: Es la función que transforma el espacio de entrada en otro espacio de características, tenemos:     
    - 'linear': Lineal
    - 'poly': Polinomial
    - 'rbf': Función de Base Radial (default)
    - 'sigmoid': Sigmoidal
   
3. **gamma**: Es el parámetro del kernel para 'rbf', 'poly' y 'sigmoid'.
   - Si gamma es 'scale' (por defecto), entonces se calcula como $ \frac{1}{{n \times X.var()}} $ para las características de X.
   - Si es 'auto', usa $ \frac{1}{n} $.

4. **degree (para kernel polinomial)**: Es el grado del polinomio.

5. **coef0 (para kernel polinomial y sigmoid)**: Término independiente en la función del kernel.

6. **shrinking**: Es un parámetro booleano. Si es verdadero, se utilizará la heurística de encogimiento, lo que puede acelerar la convergencia.**<span style="color:red">R2</span>**


**<span style="color:red">R3</span>**

In [17]:
SVC(decision_function_shape='ovr')

sklearn.svm._classes.SVC

In [108]:
# Función objetivo
def objective(params):
    clf = SVC(**params)
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    return {'loss': -acc, 'status': STATUS_OK}

# Espacio de hiper parametra;
space = {
    'C': hp.loguniform('C', -5, 2),
    'kernel': hp.choice('kernel', ['linear', 'poly', 'rbf', 'sigmoid']),
    'degree': hp.choice('degree', [2, 3, 4, 5]),
    'gamma': hp.choice('gamma', ['scale', 'auto']),
    'coef0': hp.uniform('coef0', -1, 1),
    'shrinking': hp.choice('shrinking', [True, False])
}

# Optimización
trials = Trials()
best = fmin(fn=objective, space=space, algo=tpe.suggest, max_evals=100, trials=trials)
print(best)

100%|██████████| 100/100 [09:39<00:00,  5.80s/trial, best loss: -0.9594444444444444]
{'C': 2.823757111564589, 'coef0': -0.2194803111912172, 'degree': 0, 'gamma': 0, 'kernel': 2, 'shrinking': 0}


In [113]:
best_kernel = ['linear', 'poly', 'rbf', 'sigmoid'][best['kernel']]
best_gamma = ['scale', 'auto'][best['gamma']]
best_shrinking = [True, False][best['shrinking']]

In [117]:
svc = SVC(C=best['C'],
            kernel=best_kernel,
            degree=best['degree'],
            gamma=best_gamma,
            coef0=best['coef0'],
            shrinking=best_shrinking)

svc.fit(X_train,y_train)

SVC(C=2.823757111564589, coef0=-0.2194803111912172, degree=0)

In [127]:
get_cv_scores_report_classification(svc,X_test,y_test,5)

Accuracy en CV: mean 0.93, std 0.01
Recall en CV: mean 0.93, std 0.01


In [128]:
# Realizar predicciones
y_pred = svc.predict(X_test)

# Crear una matriz de confusión
cm = confusion_matrix(y_test, y_pred)
ff.create_annotated_heatmap(z=cm).update_layout(title='Matriz de Confusión', xaxis_title='Clases Predichas', yaxis_title='Clases Actuales')

In [21]:
svc = pd.read_pickle('../02_Codes/01_Models/svc.pkl')
data = flatten_data(get_data(0))
#svc.predict(data)

In [22]:
svc.predict(data)

array([0])

In [130]:
pd.to_pickle(svc,'../02_Codes/01_Models/svc.pkl')